# E001 — Audio baseline: MFCC + GMM

Viz `experiments/E001_audio-mfcc-gmm-baseline.md` pro hypotézu a setup.

**Pipeline:**
1. Načti manifest
2. Pro každý fold (LOSO): extrahuj MFCC, aplikuj CMN, natrénuj dva GMM, score val fold
3. Vyhodnoť OOF skóre (EER, min-DCF per fold + mean ± std)
4. Zapiš výsledky do experiment logu

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf, evaluate

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

## 1. MFCC extrakce

**MFCC** (Mel-frequency cepstral coefficients) jsou standardní příznaky pro řeč.
Zachycují tvar hlasového traktu — tedy *kdo mluví*, ne *co říká*.

**CMN** (Cepstral Mean Normalization): od každého koeficientu odečteme jeho průměr
přes celou nahrávku. Tím odstraníme vliv mikrofonu a místnosti — jinak by model
rozpoznával místnost, ne člověka.

Výstup: matice `(T, 13)` kde T = počet frames (cca 100 frames/sekunda).

In [ ]:
def extract_mfcc(wav_path: Path, n_mfcc: int = 13) -> np.ndarray:
    """
    Vrať matici MFCC tvarů (T, n_mfcc) s aplikovanou CMN.
    Každý řádek = jeden frame (~10 ms).
    """
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)  # (n_mfcc, T)
    mfcc = mfcc.T                                             # (T, n_mfcc)
    mfcc -= mfcc.mean(axis=0)                                 # CMN
    return mfcc


def extract_mfcc_batch(df: "pd.DataFrame", folder: Path, n_mfcc: int = 13) -> np.ndarray:
    """
    Extrahuj MFCC pro všechny vzorky v df, vrať konkatenovanou matici (N_frames, n_mfcc).
    Taky vrať pole labels — jeden label na frame (pro trénování GMM).
    """
    all_mfcc = []
    all_labels = []
    for _, row in df.iterrows():
        # WAV je ve stejné složce jako PNG, jen jiná přípona
        for subfolder in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
            wav = folder / subfolder / (row["stem"] + ".wav")
            if wav.exists():
                break
        mfcc = extract_mfcc(wav)
        all_mfcc.append(mfcc)
        all_labels.extend([row["label"]] * len(mfcc))
    return np.vstack(all_mfcc), np.array(all_labels)


# Rychlý test na jednom vzorku
sample = manifest.iloc[0]
for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
    w = DATA / sf / (sample["stem"] + ".wav")
    if w.exists():
        break
mfcc_test = extract_mfcc(w)
print(f"Vzorek: {sample['stem']}")
print(f"MFCC shape: {mfcc_test.shape}  (frames × koeficienty)")
print(f"Délka nahrávky: ~{mfcc_test.shape[0] / 100:.1f}s")

In [ ]:
# Vizualizace MFCC — spektrogram-like pohled
fig, ax = plt.subplots(figsize=(10, 3))
img = ax.imshow(mfcc_test.T, aspect="auto", origin="lower", cmap="magma")
ax.set_xlabel("Frame")
ax.set_ylabel("MFCC koeficient")
ax.set_title(f"MFCC s CMN — {sample['stem']}")
plt.colorbar(img, ax=ax)
plt.tight_layout()
plt.show()

## 2. Trénování GMM

**GMM** (Gaussian Mixture Model) modeluje hustotu pravděpodobnosti v MFCC prostoru.
Trénujeme **dva** — jeden pro target, jeden pro non-target.

Skóre pro jeden vzorek = průměrné log-likelihood přes jeho frames:
```
score(x) = (1/T) * Σ_t [ log P(mfcc_t | GMM_target) - log P(mfcc_t | GMM_nontarget) ]
```

Proč průměr a ne suma? Aby délka nahrávky neovlivňovala skóre.

In [ ]:
def train_gmm(X: np.ndarray, n_components: int, seed: int = 67) -> GaussianMixture:
    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type="diag",  # diagonální kovariance = méně parametrů = méně overfitting
        max_iter=200,
        random_state=seed,
    )
    gmm.fit(X)
    return gmm


def score_utterance(wav_path: Path, gmm_target: GaussianMixture, gmm_nontarget: GaussianMixture) -> float:
    """LLR skóre pro jednu nahrávku. Vyšší = více target."""
    mfcc = extract_mfcc(wav_path)
    llr = gmm_target.score_samples(mfcc) - gmm_nontarget.score_samples(mfcc)
    return float(llr.mean())

## 3. Cross-validation — OOF skóre

Pro každý fold:
1. Train fold → extrahuj MFCC → natrénuj GMM_target a GMM_nontarget
2. Val fold → pro každý vzorek spočítej LLR skóre
3. Ulož do `oof_scores`

**Důležité:** CMN počítáme per-utterance (uvnitř `extract_mfcc`), takže nedochází k úniku z val do train.

In [ ]:
N_MFCC = 13
N_TARGET_COMPONENTS = 8
N_NONTARGET_COMPONENTS = 32
SEED = 67

oof_scores = np.full(len(manifest), np.nan)
fold_results = []

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]

    print(f"Fold {fold_id}: train={len(train_df)}, val={len(val_df)} "
          f"(target val: {(val_df.label==1).sum()}, non-target val: {(val_df.label==0).sum()})")

    # Extrakce příznaků — pouze z train foldu
    print(f"  Extrahuji MFCC pro train...")
    X_train, y_train = extract_mfcc_batch(train_df, DATA)

    X_target    = X_train[y_train == 1]
    X_nontarget = X_train[y_train == 0]
    print(f"  Train frames: {len(X_target)} target, {len(X_nontarget)} non-target")

    # Trénování GMM
    print(f"  Trénuji GMM...")
    gmm_t  = train_gmm(X_target,    n_components=N_TARGET_COMPONENTS,    seed=SEED)
    gmm_nt = train_gmm(X_nontarget, n_components=N_NONTARGET_COMPONENTS, seed=SEED)

    # Skórování val foldu
    print(f"  Skóruji val fold...")
    for idx, row in val_df.iterrows():
        for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
            wav = DATA / sf / (row["stem"] + ".wav")
            if wav.exists():
                break
        oof_scores[idx] = score_utterance(wav, gmm_t, gmm_nt)

    # Metriky pro tento fold
    val_scores = oof_scores[val_idx]
    val_labels = manifest.loc[val_idx, "label"].to_numpy()
    eer, _     = compute_eer(val_scores[val_labels==1], val_scores[val_labels==0])
    min_dcf, _ = compute_min_dcf(val_scores[val_labels==1], val_scores[val_labels==0])
    fold_results.append({"fold": fold_id, "eer": eer, "min_dcf": min_dcf})
    print(f"  → EER = {eer*100:.2f}%, min-DCF = {min_dcf:.4f}\n")

print("Hotovo.")

## 4. Výsledky

In [ ]:
import pandas as pd

results_df = pd.DataFrame(fold_results)
mean_eer    = results_df["eer"].mean()
std_eer     = results_df["eer"].std()
mean_dcf    = results_df["min_dcf"].mean()
std_dcf     = results_df["min_dcf"].std()

print("=" * 45)
print(f"{'Fold':<8} {'EER [%]':>10} {'min-DCF':>10}")
print("-" * 45)
for _, r in results_df.iterrows():
    print(f"{int(r.fold):<8} {r.eer*100:>10.2f} {r.min_dcf:>10.4f}")
print("-" * 45)
print(f"{'mean±std':<8} {mean_eer*100:>7.2f}±{std_eer*100:.2f} {mean_dcf:>7.4f}±{std_dcf:.4f}")
print("=" * 45)

# OOF celkové
y_all = manifest["label"].to_numpy()
eer_oof, _     = compute_eer(oof_scores[y_all==1], oof_scores[y_all==0])
dcf_oof, thr   = compute_min_dcf(oof_scores[y_all==1], oof_scores[y_all==0])
print(f"\nOOF celkové: EER = {eer_oof*100:.2f}%, min-DCF = {dcf_oof:.4f}, threshold = {thr:.3f}")

In [ ]:
# Vizualizace distribucí OOF skóre
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
bins = np.linspace(oof_scores.min(), oof_scores.max(), 40)
ax.hist(oof_scores[y_all==0], bins=bins, alpha=0.6, color="steelblue", label="non-target", density=True)
ax.hist(oof_scores[y_all==1], bins=bins, alpha=0.7, color="tomato",    label="target",     density=True)
ax.axvline(thr, color="green", ls="--", lw=2, label=f"threshold = {thr:.2f}")
ax.set_xlabel("OOF skóre (LLR)")
ax.set_title("Distribuce skóre")
ax.legend()

from sklearn.metrics import roc_curve, auc
ax = axes[1]
fpr, tpr, _ = roc_curve(y_all, oof_scores)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, color="tomato", lw=2, label=f"AUC = {roc_auc:.3f}")
ax.plot([0,1],[0,1],"k--",lw=1)
ax.set_xlabel("FAR")
ax.set_ylabel("1 - FRR")
ax.set_title("ROC křivka (OOF)")
ax.legend()

plt.suptitle(f"E001 — MFCC+GMM baseline  |  EER = {eer_oof*100:.2f}%, min-DCF = {dcf_oof:.4f}")
plt.tight_layout()
plt.show()

## 5. Zapiš výsledky do experiment logu

Zkopíruj čísla z výše do `experiments/E001_audio-mfcc-gmm-baseline.md`:
- tabulka fold výsledků
- interpretace (co překvapilo, jak se foldy liší)
- next step (co z toho plyne)